In [3]:
import os
import json
import logging
from urllib.parse import urlparse
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
import torchvision.transforms as transforms

# 기본 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- 설정 (사용자 환경에 맞게 수정하세요) ---
JSON_FILE_PATH = './animal_data.json'
UPDATED_JSON_FILE_PATH = './animal_data_updated_min.json'
IMAGES_DIR = './images'
MODEL_WEIGHT_PATH = './FlaskProject-Deep-Bark/model/best_ef_b5_min (1).pth'
VISUALIZATION_FILE_PATH = './breed_update_visualization.png'

# 모델이 학습한 품종 이름 목록
CLASS_NAMES = sorted([
    'Beagle', 'Bichon Frise', 'Border Collie', 'ChowChow', 'Chihuahua',
    'Cocker Spaniel', 'Dachshund', 'Doberman', 'Golden Retriever', 'Jindo Dog', 'Maltese dog',
    'Pembroke Welsh Corgi', 'Pomeranian dog', 'Pug', 'Samoyed dog', 'Shiba Inu', 'Shih Tzu',
    'Siberian Husky', 'Toy Poodle', 'Yorkshire Terrier'
])
# ---------------------------------------------

# =============================================
#  강아지 품종 예측 모델 함수 (breed_model.py 내용)
# =============================================
def load_model(model_path, num_labels):
    """
    학습된 EfficientNet-B5 모델을 로드합니다.
    """
    device = torch.device(
        "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

    # model = models.efficientnet_b5(weights=None)
    model = models.resnet50(weights=None)

    in_features = model.classifier[1].in_features
    logger.info(f"모델 분류기의 입력 특성: {in_features}")
    model.classifier[1] = nn.Linear(in_features, num_labels)

    model.load_state_dict(torch.load(model_path, map_location=device), strict=False)

    model.to(device)
    model.eval()
    return model, device


def predict_image(image_path, model, device, label_names):
    """
    Sigmoid 함수를 사용하여 이미지의 품종을 예측합니다.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.sigmoid(outputs)[0]
        predicted_label = [label_names[probs.argmax()]]
        class_probs = {label_names[i]: float(prob.item()) * 100 for i, prob in enumerate(probs)}

        return predicted_label, class_probs

# =============================================
#  JSON 업데이트 및 시각화 함수
# =============================================
def find_and_update_breed(info_list, new_breed):
    """ info 리스트에서 '품종'을 찾아 값을 업데이트하고 원본 품종을 반환합니다. """
    for sublist in info_list:
        try:
            index = sublist.index('품종')
            if index + 1 < len(sublist):
                original_breed = sublist[index + 1]
                sublist[index + 1] = new_breed
                logger.info(f"품종 업데이트: '{original_breed}' -> '{new_breed}'")
                return original_breed, True
        except ValueError:
            continue
    return None, False

def create_visualization(original_breeds, predicted_breeds):
    """ 원본 품종과 예측된 품종의 분포를 막대그래프로 시각화하고 파일로 저장합니다. """
    if not original_breeds and not predicted_breeds:
        logger.info("시각화할 데이터가 없습니다.")
        return

    try:
        plt.rc('font', family='AppleGothic') # macOS
    except:
        logger.warning("한글 지원 폰트(Malgun Gothic, AppleGothic)를 찾을 수 없어 그래프의 한글이 깨질 수 있습니다.")

    plt.rcParams['axes.unicode_minus'] = False

    df = pd.DataFrame({
        'original': pd.Series(original_breeds).value_counts(),
        'predicted': pd.Series(predicted_breeds).value_counts()
    }).fillna(0).astype(int)

    df.plot(kind='bar', figsize=(18, 10), width=0.8)
    plt.title('원본 품종 vs AI 예측 품종 분포 비교', fontsize=20)
    plt.xlabel('품종', fontsize=14)
    plt.ylabel('개체 수', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.legend(['원본', 'AI 예측'])
    plt.tight_layout()

    plt.savefig(VISUALIZATION_FILE_PATH)
    logger.info(f"시각화 결과가 '{VISUALIZATION_FILE_PATH}' 파일로 저장되었습니다.")
    plt.close()


def main():
    """
    JSON 파일을 읽고, 이미지를 예측하여 '품종' 필드를 업데이트한 후,
    결과를 새 JSON 파일과 시각화 차트로 저장하는 메인 함수입니다.
    """
    logger.info(f"'{MODEL_WEIGHT_PATH}' 에서 모델을 로드합니다...")
    if not os.path.exists(MODEL_WEIGHT_PATH):
        logger.error(f"모델 가중치 파일을 찾을 수 없습니다: {MODEL_WEIGHT_PATH}")
        return

    try:
        model, device = load_model(MODEL_WEIGHT_PATH, len(CLASS_NAMES))
        logger.info("모델 로딩 완료.")
    except Exception as e:
        logger.error(f"모델 로딩 중 오류 발생: {e}")
        return

    logger.info(f"'{JSON_FILE_PATH}' 에서 JSON 데이터를 로드합니다...")
    try:
        with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
            animal_data = json.load(f)
        logger.info(f"총 {len(animal_data)}개의 동물 데이터를 로드했습니다.")
    except FileNotFoundError:
        logger.error(f"JSON 파일을 찾을 수 없습니다: {JSON_FILE_PATH}")
        return
    except json.JSONDecodeError:
        logger.error(f"JSON 파일 형식이 올바르지 않습니다: {JSON_FILE_PATH}")
        return

    updated_count = 0
    original_breeds_list = []
    predicted_breeds_list = []

    for i, animal in enumerate(animal_data):
        image_urls = animal.get('images')
        if not image_urls or not isinstance(image_urls, list) or len(image_urls) == 0:
            logger.warning(f"항목 {i+1}/{len(animal_data)}: 'images' 필드가 없어 건너뜁니다.")
            continue

        image_url = image_urls[0]
        image_filename = os.path.basename(urlparse(image_url).path)
        image_path = os.path.join(IMAGES_DIR, image_filename)

        if not os.path.exists(image_path):
            logger.warning(f"항목 {i+1}/{len(animal_data)}: 로컬 이미지 파일을 찾을 수 없습니다 - {image_path}. 건너뜁니다.")
            continue

        try:
            predicted_label, _ = predict_image(image_path, model, device, CLASS_NAMES)
            predicted_breed = predicted_label[0] if predicted_label else "판독 불가"

            original_breed, updated = find_and_update_breed(animal.get('info', []), predicted_breed)

            if updated:
                updated_count += 1
                original_breeds_list.append(original_breed)
                predicted_breeds_list.append(predicted_breed)
            else:
                logger.warning(f"항목 {i+1}/{len(animal_data)}: 'info' 리스트에서 '품종' 필드를 찾지 못했습니다.")

        except Exception as e:
            logger.error(f"이미지 '{image_path}' 예측 중 오류 발생: {e}")

    logger.info(f"총 {updated_count}개의 항목을 업데이트했습니다. '{UPDATED_JSON_FILE_PATH}' 파일로 저장합니다...")
    try:
        with open(UPDATED_JSON_FILE_PATH, 'w', encoding='utf-8') as f:
            json.dump(animal_data, f, ensure_ascii=False, indent=2)
        logger.info("성공적으로 저장되었습니다.")
    except Exception as e:
        logger.error(f"업데이트된 JSON 파일 저장 중 오류 발생: {e}")

    create_visualization(original_breeds_list, predicted_breeds_list)

if __name__ == '__main__':
    main()

2025-09-16 19:25:40,822 - INFO - './FlaskProject-Deep-Bark/model/best_ef_b5_min (1).pth' 에서 모델을 로드합니다...
2025-09-16 19:25:41,012 - ERROR - 모델 로딩 중 오류 발생: 'ResNet' object has no attribute 'classifier'
